# EMNIST: полная сетка экспериментов для v5 (SE-CNN)

**Цель:** доказать превосходство v5 во всех конфигурациях аугментация × scheduler.

**Сетка:** v5 × 3 аугментации × 3 scheduler = **9 запусков**

**Параметры:** 25 эпох, seed=42, batch=64, lr=1e-3, AdamW + label_smoothing=0.1

**Ожидаемое время:** ~3ч 45мин на T4.

## 1. Клонирование репозитория и установка зависимостей

In [ ]:
REPO_URL = 'https://github.com/sultonovmuhibulloh612/emnist-master-thesis.git'

import os
if not os.path.exists('/content/emnist-project'):
    !git clone $REPO_URL /content/emnist-project
%cd /content/emnist-project

!pip install -q tensorboard seaborn

## 2. Проверка GPU

In [ ]:
import torch
print(f'CUDA доступна: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
!nvidia-smi

## 3. ⚠️ ВАЖНО: проверь, что в train.py применены патчи и есть v5

Перед запуском в `src/training/train.py` должно быть:
1. `criterion = nn.CrossEntropyLoss(label_smoothing=0.1)`
2. `optimizer = optim.AdamW(...)`
3. В реестре моделей добавлен `improved_cnn_v5`
4. В `choices` парсера тоже добавлен v5

Также проверь, что в `src/models/` лежит `improved_cnn_v5.py`.

In [ ]:
# Проверка наличия файла v5 и патчей в train.py
import os

v5_exists = os.path.exists('src/models/improved_cnn_v5.py')
print(f'{"✓" if v5_exists else "✗ ОТСУТСТВУЕТ"} src/models/improved_cnn_v5.py')

with open('src/training/train.py', encoding='utf-8') as f:
    code = f.read()

checks = {
    'AdamW в оптимизаторе':  'optim.AdamW' in code,
    'label_smoothing=0.1':    'label_smoothing=0.1' in code,
    'v5 в реестре моделей':   'improved_cnn_v5' in code,
}
print()
for desc, ok in checks.items():
    print(f'{"✓" if ok else "✗"} {desc}')

## 4. Запуск 9 экспериментов: v5 × 3 аугментации × 3 scheduler

In [ ]:
import subprocess, time
from datetime import datetime

MODEL = 'improved_cnn_v5'
AUGMENTATIONS = ['none', 'light', 'strong']
SCHEDULERS    = ['step', 'cosine', 'plateau']
SEED          = 42

EPOCHS, BATCH_SIZE, LR, WEIGHT_DECAY, SPLIT = 25, 64, 1e-3, 1e-4, 'balanced'

configs = [(aug, sch) for aug in AUGMENTATIONS for sch in SCHEDULERS]
total = len(configs)

print(f'Запусков: {total}')
print(f'Старт: {datetime.now().strftime("%H:%M:%S")}')
print('=' * 70)

t_start = time.time()
results = []

for run_idx, (aug, sch) in enumerate(configs, 1):
    exp_name = f'v5_{aug}_{sch}'

    cmd = [
        'python', '-m', 'src.training.train',
        '--model',        MODEL,
        '--experiment',   exp_name,
        '--epochs',       str(EPOCHS),
        '--batch_size',   str(BATCH_SIZE),
        '--lr',           str(LR),
        '--weight_decay', str(WEIGHT_DECAY),
        '--augmentation', aug,
        '--scheduler',    sch,
        '--split',        SPLIT,
        '--seed',         str(SEED),
        '--num_workers',  '2',
    ]

    print(f'\n[{run_idx}/{total}] {exp_name}')
    print(f'  Старт: {datetime.now().strftime("%H:%M:%S")}')
    t0 = time.time()

    result = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.time() - t0
    status = 'OK' if result.returncode == 0 else 'FAILED'
    print(f'  Статус: {status} | Время: {elapsed/60:.1f} мин')

    if result.returncode != 0:
        print('  Последние строки stderr:')
        print('  ' + '\n  '.join(result.stderr.strip().split('\n')[-5:]))

    results.append({
        'experiment': exp_name,
        'status':     status,
        'time_min':   elapsed / 60,
    })

print('\n' + '=' * 70)
total_min = (time.time() - t_start) / 60
print(f'ВСЕГО ВРЕМЕНИ: {total_min:.1f} мин ({total_min/60:.2f} ч)')
print(f'OK: {sum(1 for r in results if r["status"] == "OK")}/{total}')
print('\nСводка:')
for r in results:
    print(f"  {r['status']:6} | {r['time_min']:5.1f} мин | {r['experiment']}")

## 5. Архивация результатов

Делаем **два архива:**
- **Полный** (со всеми файлами, включая .pth модели и tensorboard) — для тебя
- **Лёгкий** (без .pth и tensorboard, только CSV/JSON/PNG) — для Claude на анализ

### 5.1. Полный архив для тебя (со всеми файлами)

In [ ]:
# Полный архив — все файлы экспериментов v5
!cd /content/emnist-project && \
  tar -czvf v5_grid_full.tar.gz \
  results/*v5_none* results/*v5_light* results/*v5_strong* 2>&1 | tail -5

!ls -lh /content/emnist-project/v5_grid_full.tar.gz

### 5.2. Лёгкий архив для Claude (с exclude'ами)

In [ ]:
# Лёгкий архив — без .pth моделей и tensorboard событий
!cd /content/emnist-project && \
  tar --exclude='*.pth' --exclude='*.pt' \
      --exclude='events.out.tfevents.*' --exclude='tensorboard' \
      -czvf v5_grid_light.tar.gz \
      results/*v5_none* results/*v5_light* results/*v5_strong* 2>&1 | tail -5

!ls -lh /content/emnist-project/v5_grid_light.tar.gz

## 6. Скачивание архивов

### 6.1. Скачать полный архив для себя

In [ ]:
from google.colab import files
files.download('/content/emnist-project/v5_grid_full.tar.gz')

### 6.2. Скачать лёгкий архив для Claude

In [ ]:
from google.colab import files
files.download('/content/emnist-project/v5_grid_light.tar.gz')

## 7. (Опционально) Сохранить в Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/emnist-results/
!cp /content/emnist-project/v5_grid_full.tar.gz  /content/drive/MyDrive/emnist-results/
!cp /content/emnist-project/v5_grid_light.tar.gz /content/drive/MyDrive/emnist-results/
print('✓ Сохранено в Google Drive: /MyDrive/emnist-results/')